# Generowanie danych do plików csv

## Użyte biblioteki

In [2]:
from faker import Faker
import random
from countryinfo import CountryInfo
import csv

## Generowanie informacji o ludziach z podziałem na studentów i pracowników

Generator gwarantuje unikatowość identyfikatotów, adresów email oraz numerów telefonów. Obecna implementacja umożliwia generowanie danych dla różnych krajów.

In [3]:
class Person:
    
    student_counter = 1000
    employee_counter = 1000
    generated_phone_numbers = set()
    generated_emails = set()
    
    def __init__(self, position, symbol):
        self.firstNameGenerate(symbol)
        self.lastNameGenerate(symbol)
        self.status = position
        self.birthDateGenerate()
        self.cityGenerate(symbol)
        self.streetAddressGenerate(symbol)
        self.getCountryName(symbol)
        self.getStudentID()
        self.getEmployeeID()
        self.phoneNumberGenerate(symbol)
        self.emailGenerate()
        self.jobPosition = None
    
    def firstNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.firstName = fake.first_name()
    
    def lastNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.lastName = fake.last_name()
        
    def cityGenerate(self, symbol):
        fake = Faker(symbol)
        self.city = fake.city()
    
    def streetAddressGenerate(self, symbol):
        fake = Faker(symbol)
        self.streetAddress = fake.street_address()
    
    def getCountryName(self, symbol):
        
        country_dict = {
            "US": "Stany Zjednoczone",
            "PL": "Polska",
            "DE": "Niemcy",
            "GB": "Wielka Brytania",
            "FR": "Francja",
            "IT": "Włochy"
        }
        
        country_code = symbol.split('_')[1]
        self.country = country_dict[country_code]
    
    
    def birthDateGenerate(self):
        position = self.status
        fake = Faker()
        match position:
            case 'student':
                age_ranges = [(18, 25), (26, 30), (31, 50), (51, 65)]
                weights = [0.6, 0.2, 0.15, 0.05]
            case 'pracownik':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.2, 0.4, 0.3, 0.1]
            
        selected_range = random.choices(age_ranges, weights=weights, k=1)[0]
        min_age, max_age = selected_range
        self.birthDate = fake.date_of_birth(None, min_age, max_age)
    
    def getStudentID(self):
        if self.status == 'student':
            self.studentID = Person.student_counter
            Person.student_counter += 1
        else:
            self.studentID = None
    
    def getEmployeeID(self):
        if self.status == 'pracownik':
            self.employeeID = Person.employee_counter
            Person.employee_counter += 1
        else:
            self.employeeID = None 
    
    def phoneNumberGenerate(self, symbol):
        faker = Faker(symbol)
        country_code = symbol.split('_')[1]
        informations = CountryInfo(country_code)
        calling_code = informations.calling_codes()[0]
        
        while True:
            phone_number = faker.phone_number()
            if not phone_number[0] == '+':
                phone_number = "+" + calling_code + " " + phone_number
            if phone_number not in Person.generated_phone_numbers:
                self.phone = phone_number
                Person.generated_phone_numbers.add(phone_number)
                break
    
    def emailGenerate(self):
        domains = [
            "gmail.com",
            "outlook.com",
            "interia.pl",
            "yahoo.com",
            "wp.pl"
        ]
        weights = [0.4, 0.1, 0.2, 0.1, 0.2]
        trans = str.maketrans("ąćęłńóśźż", "acelnoszz")
        first_name = self.firstName.translate(trans)
        last_name = self.lastName.translate(trans)
        
        while True:
            domain =  random.choices(domains, weights=weights, k=1)[0]
            random_number = random.randint(1, 9999)
            random_separator = random.choice([".", "_", "-"])
            
            email_prefix = random.choice([
            f"{first_name}{random_number}{last_name}",
            f"{first_name}{random_separator}{last_name}",
            f"{last_name}{random_separator}{first_name}",
            f"{last_name}{random_separator}{first_name}{random_number}",
            f"{first_name}{random_number}"
            f"{last_name}{random_separator}{random_number}"
            ]).lower()
            
            email = email_prefix + "@" + domain
            if email not in Person.generated_emails:
                self.email = email
                Person.generated_emails.add(email)
                break

### Przykład użycia

In [4]:

person = Person('pracownik', 'pl_PL')
print('First name:', person.firstName)
print('Last name:', person.lastName)
print('Position:', person.status)
print('Birth Date:', person.birthDate)
print('City:', person.city)
print('Address:', person.streetAddress)
print('Country:', person.country)
print('Employee ID:', person.employeeID)
print('Phone number:', person.phone)
print('Email:', person.email)


First name: Julita
Last name: Dąbrowicz
Position: pracownik
Birth Date: 1975-09-13
City: Otwock
Address: ul. Wybickiego 84
Country: Polska
Employee ID: 1000
Phone number: +48 780 632 162
Email: dabrowicz_julita7302@gmail.com


### Zapisywanie danych do pliku .csv

In [5]:
def SavetoCsv(filename, data):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        for row in data:
            writer.writerow(row)

### Przykładowe generowanie danych dla studentów i zapisanie ich

In [6]:
students_info = []
for i in range(100):
    student = Person('student', 'pl_PL')
    student_data = [student.studentID, 
                    student.firstName, 
                    student.lastName, 
                    student.birthDate, 
                    student.country, 
                    student.city, 
                    student.streetAddress, 
                    student.email, 
                    student.phone
                ]
    students_info.append(student_data)
SavetoCsv('student.csv', students_info)

## Generowanie informacji na temat sal

In [63]:
class LectureRoom:
    id_counter = 100
    lecture_rooms = set()
    
    def __init__(self):
        self.id = LectureRoom.id_counter
        self.buildingInfoGenerate()
        LectureRoom.id_counter += 1
    
    def buildingInfoGenerate(self):
        buildings = ['A-0', 'A-1', 'A-2', 'B-1', 'B-2', 'C-1', 'C-2', 'C-3']
        while True:
            floor = random.randint(1, 4)
            class_number = random.randint(1, 20)
            building = random.choice(buildings)
            new_class = str(class_number) + " " + str(floor) + building
            
            if new_class not in LectureRoom.lecture_rooms:
                self.building = building
                self.floor = floor
                self.classNumber = class_number
                LectureRoom.lecture_rooms.add(new_class)
                break

### Przykład generowanych sal lekcyjnych

In [64]:
classes = [LectureRoom(), LectureRoom(), LectureRoom()]

for i in range(3):
    print(f"\nSala {i}")
    print('Room ID:', classes[i].id)
    print('Building:', classes[i].building)
    print('Floor:', classes[i].floor)
    print('Class number:', classes[i].classNumber)


Sala 0
Room ID: 100
Building: B-2
Floor: 4
Class number: 13

Sala 1
Room ID: 101
Building: C-3
Floor: 2
Class number: 2

Sala 2
Room ID: 102
Building: A-2
Floor: 3
Class number: 7


### Przykładowe generowanie danych dla sal i zapisanie ich

In [65]:
rooms_info = []
for i in range(100):
    room = LectureRoom()
    room_data = [
                    room.id,
                    room.building, 
                    room.floor, 
                    room.classNumber
                ]
    rooms_info.append(room_data)
SavetoCsv('lecturerooms.csv', rooms_info)